In [44]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier,DecisionTreeRegressor,export_text
from sklearn.model_selection import train_test_split,GridSearchCV,learning_curve
from sklearn.metrics import accuracy_score,roc_auc_score,classification_report
from sklearn.datasets import make_classification,make_regression
import warnings; warnings.filterwarnings("ignore")

In [45]:
np.random.seed(42)

#### Gini vs Entropy


In [46]:
def gini(p:float)->float:
  return 1-(p**2+(1-p)**2)

def entropy(p:float)->float:
  if p==0 or p==1:
    return 0
  return -(p*np.log2(p)+(1-p)*np.log2(1-p))

In [47]:
print('At p=0.5 (Max uncertainty): Gini={:.3f},Entropy={:.3f}'.format(gini(0.5),entropy(0.5)))
print(f'At p=0.9 (mostly one class): Gini={gini(0.9):.3f},Entropy={entropy(0.9):.3f}')
print(f'At p=0.1 (mostly other class): Gini={gini(0.01):.3f},Entropy={entropy(0.01):.3f}')

At p=0.5 (Max uncertainty): Gini=0.500,Entropy=1.000
At p=0.9 (mostly one class): Gini=0.180,Entropy=0.469
At p=0.1 (mostly other class): Gini=0.020,Entropy=0.081


#### Compute Information Gain manually

In [48]:
parent_y=np.array([1,1,1,0,0,0,0,0,1,0])
p_parent=parent_y.mean()

#After split on some features
left_y=np.array([1,1,1,1,0])
right_y=np.array([0,0,0,0,0])

parent_entropy=entropy(p_parent)
left_entropy=entropy(left_y.mean())
right_entropy=entropy(right_y.mean())

n,n_left,n_right=len(parent_y),len(left_y),len(right_y)
info_gain=parent_entropy-((n_left/n*left_entropy)+(n_right/n*right_entropy))

print(f'''Parent entropy: {parent_entropy:.3f}
Left entropy: {left_entropy:.3f}
Right entropy: {right_entropy:.3f}''')

print(f'Information gain split:{info_gain:.4f} <- perfect split -> max IG!')

Parent entropy: 0.971
Left entropy: 0.722
Right entropy: 0.000
Information gain split:0.6100 <- perfect split -> max IG!


In [49]:
X,y=make_classification(n_samples=1000,n_features=10,n_informative=5,random_state=42)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [51]:
print(f"{'Max Depth':>10} {'Train Acc':>12}  {'Test Acc':>12}  {'Diagnosis':>12}")
for depth in [1,2,4,6,8,10,12,None]:
  dt=DecisionTreeClassifier(max_depth=depth,random_state=42)
  dt.fit(X_train,y_train)
  train_acc=dt.score(X_train,y_train)
  test_acc=dt.score(X_test,y_test)
  gap=train_acc-test_acc
  diagnosis=('Underfitting' if train_acc < 0.75
             else "Overfitting" if gap > 0.15
             else "Good fit" if gap > 0.05
             else "Slight overfit")
  print(f"{str(depth):>10} {train_acc:>12.3f} {test_acc:>12.3f} {diagnosis:>20}")


 Max Depth    Train Acc      Test Acc     Diagnosis
         1        0.709        0.770         Underfitting
         2        0.814        0.845       Slight overfit
         4        0.917        0.925       Slight overfit
         6        0.966        0.910             Good fit
         8        0.996        0.895             Good fit
        10        1.000        0.895             Good fit
        12        1.000        0.895             Good fit
      None        1.000        0.895             Good fit


In [57]:
param_grid={
    'max_depth':[3,5,7,10],
    'min_samples_leaf':[1,5,10,20],
    'criterion':['gini','entropy'],
    'ccp_alpha':[0.0,0.001,0.005]
}

grid_search=GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
    refit=True
)
grid_search.fit(X_train,y_train)
print(f'''
Best parameters: {grid_search.best_params_}
Best CV AUC:{grid_search.best_score_:4f}
Test AUC:{roc_auc_score(y_test,grid_search.predict_proba(X_test)[:,1]):.4f}
''')
best_tree=grid_search.best_estimator_


Best parameters: {'ccp_alpha': 0.001, 'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 10}
Best CV AUC:0.938977
Test AUC:0.9397



### Feature Importance

In [58]:
feature_names=[f"feature_{i}"for i in range(X.shape[1])]
importance_df=pd.DataFrame({
    'feature':feature_names,
    "importance":best_tree.feature_importances_
}).sort_values("importance",ascending=False)
print(importance_df.to_string())


     feature  importance
0  feature_0    0.296174
5  feature_5    0.277059
4  feature_4    0.275101
3  feature_3    0.077256
6  feature_6    0.054558
9  feature_9    0.014173
2  feature_2    0.003084
7  feature_7    0.002597
1  feature_1    0.000000
8  feature_8    0.000000


In [59]:
train_sizes,train_scores,val_scores=learning_curve(
    DecisionTreeClassifier(max_depth=5,random_state=42),
    X,y,
    train_sizes=np.linspace(0.1,1,10),
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

In [68]:
print(f'''=== Learning Curve(max_depth=5) ===
{'Train size':>12}   {'Train acc':>12}  {'Val acc':>12} {'Verdict':>15}
''')
for i,sz in enumerate(train_sizes):
  tr=train_scores[i].mean()
  vl=val_scores[i].mean()
  gap=tr-vl
  verdict="High variance" if gap >0.1 else 'Good' if gap <0.03 else "Ok"
  print(f"  {int(sz):>10}     {tr:>10.3f}    {vl:>10.3f}       {verdict:>15}")

=== Learning Curve(max_depth=5) ===
  Train size      Train acc       Val acc         Verdict

          80          1.000         0.781         High variance
         160          0.990         0.806         High variance
         240          0.968         0.859         High variance
         320          0.960         0.876                    Ok
         400          0.964         0.877                    Ok
         480          0.957         0.882                    Ok
         560          0.948         0.871                    Ok
         640          0.950         0.876                    Ok
         720          0.949         0.873                    Ok
         800          0.937         0.880                    Ok


In [69]:
small_tree=DecisionTreeClassifier(max_depth=2,random_state=42)
small_tree.fit(X_train,y_train)
print(f'''Tree strcuture (max_depth=2):
{export_text(small_tree,feature_names=feature_names[:X.shape[1]])}''')

Tree strcuture (max_depth=2):
|--- feature_4 <= -0.20
|   |--- feature_3 <= -1.00
|   |   |--- class: 0
|   |--- feature_3 >  -1.00
|   |   |--- class: 1
|--- feature_4 >  -0.20
|   |--- feature_5 <= -0.28
|   |   |--- class: 1
|   |--- feature_5 >  -0.28
|   |   |--- class: 0

